In [ ]:
import sys, ohlcv_lab                                                                                        
print(sys.executable)                                   
print(ohlcv_lab.__file__)

In [ ]:
import numpy as np                                                                                           
import pandas as pd                                                                                          
from ohlcv_lab.clean import read_raw, clean                                                                  
from ohlcv_lab.storage import load_clean  

In [ ]:
raw = read_raw("../data/raw", "BTCUSDT", "60")   # путь относительно папки notebooks/                        
raw.head()                                                                                                   
raw.dtypes        # всё строки (str) — так пришло из API                                                     
raw.info()        # число строк, типы, память  

In [ ]:
# Портим данные специально:                                                                          
bad = raw.copy()                                                                                             
bad = pd.concat([bad, bad.iloc[[100, 200]]])   # 2 точных дубля                                              
bad = bad.drop(index=range(500, 505))          # 5 пропущенных часов                                         
bad.loc[10, "close"] = None                    # одно пустое значение                                        
len(raw), len(bad)   

In [ ]:
bad.isnull()                  # таблица True/False того же размера                                           
bad.isnull().sum()            # сколько пустых в каждой колонке                                              
bad[bad["close"].isnull()]    # сами строки с пустым close                                                   
bad.isnull().any(axis=1).sum()  # сколько строк, где пусто хоть что-то

In [ ]:
bad.duplicated().sum()                         # полные дубли строк                                          
bad.duplicated(subset=["open_time"]).sum()     # дубли только по времени                                     
bad[bad.duplicated(subset=["open_time"], keep=False)]   # показать ВСЕ копии, а не только повторы            
bad.drop_duplicates().shape 

# display(s)

In [ ]:
bad.isnull().sum()["open_time"]    # 0 — а 5 часов пропало!                                                  

ts = pd.to_datetime(bad["open_time"].astype("int64"), unit="ms", utc=True)                                   
s = bad.assign(open_time=ts).drop_duplicates("open_time").set_index("open_time").sort_index()                
grid = pd.date_range(s.index.min(), s.index.max(), freq="h")                                                 
full = s.reindex(grid)                                                                                       
full["close"].isnull().sum()       # теперь пропуски видны (5 + тот, что мы обнулили)                        
full[full["close"].isnull()] 